# King+Rook vs King+Pawn on A7 (KRKPA7)

**Autores:** Valamy Donado y Abraham Navarro  
**Fecha:** 2026-08-23

---

## Introducción

El ajedrez, además de ser un deporte, constituye un dominio clásico para el estudio de la toma de decisiones y el razonamiento automático (Russell y Norvig, 2020). Desde los años cincuenta, se han utilizado los finales de ajedrez como pruebas para algoritmos de aprendizaje automático, ya que poseen un número reducido de piezas, reglas perfectamente definidas y resultados verificables objetivamente: victoria, derrota o tablas (Shapiro, 1983). 

Tradicionalmente se reconocen tres fases en una partida de ajedrez: la apertura, donde predomina el desarrollo de las piezas y el control del centro; el medio juego, donde se producen maniobras de ataque y defensa contra el rey rival o sus debilidades estructurales; y el final, donde tras varios intercambios la promoción de peones se convierte en el tema dominante (Dvoretsky, 2003). Uno de los aspectos más fascinantes del ajedrez es el estudio de los finales, donde el número de piezas se reduce y el juego se vuelve más teórico y preciso (Nunn, 2010). Uno de ellos es el final Rey + Torre vs Rey + Peón, o KRKPA7 (**K**ing + **R**ook vs **K**ing + **P**awn on **a7**).

## Objetivos

Objetivo general: Analizar el comportamiento del final de ajedrez Rey + Torre vs Rey + Peón a partir del dataset KRKPA7, mediante técnicas de análisis exploratorio de datos, visualización interactiva y modelado predictivo, para identificar los patrones que determinan si el peón (negras) logra ganar o es detenido por las blancas.

Objetivos específicos:
- Realizar un análisis que caracterice de forma univariada y bivariada las 36 variables categóricas del dataset, identificando su distribución, asociación con la variable objetivo y relaciones entre predictoras, mediante tablas de frecuencia, matrices de asociación (V de Cramér) y visualizaciones interactivas (Cramér, 1946).
- Construir un modelo predictivo que, a partir de las variables del tablero, que clasifique correctamente si una posición termina en victoria del bando del peón, es decir, las negras (won), o no (nowin), evaluando su desempeño mediante métricas como exactitud, matriz de confusión, F1-score, AUC-ROC, y validación cruzada.

## Marco teórico

### Finales de ajedrez

Los finales de ajedrez han sido estudiados desde dos perspectivas complementarias: la teoría ajedrecística, que establece reglas prácticas sobre cómo ganar o entablar, y la teoría de la computación, que los utiliza como problemas de búsqueda y clasificación (Russell y Norvig, 2020), estos, ya con pocas piezas, permiten definir con precisión qué significa "ganar" y "no ganar" en cada posición, y por lo tanto admiten reglas verificables que pueden ser aprendidas, inducidas o programadas. La teoría clásica, recogida por ejemplo en el manual de finales de Dvoretsky, establece que las blancas ganan si logran detener al peón con la torre y acercar su rey para dar mate. Las negras, en cambio, obtienen tablas o ganan si coronan el peón, ahogan al rey blanco o capturan la torre. 

### Medidas de asociación entre variables categóricas

Dado que todas las variables del dataset KRKPA7 son de naturaleza categórica, no es posible utilizar coeficientes de correlación como el de Pearson o Spearman, que están diseñados para variables cuantitativas. En su lugar, se emplean medidas de asociación específicas para datos cualitativos, siendo la más utilizada en la literatura la V de Cramér, propuesta por Harald Cramér en 1946 (Cramér, 1946). Es una medida estadística que cuantifica la fuerza de la asociación entre dos variables categóricas. Sus valores oscilan entre 0 y 1. Un valor de 0 indica que no existe relación entre las variables, es decir, que son independientes. Un valor de 1 indica una relación perfecta, en la que una variable determina completamente a la otra. Valores intermedios sugieren distintos grados de asociación, donde valores cercanos a 0.1 se consideran asociaciones débiles, cercanos a 0.3 moderadas, cercanos a 0.5 fuertes y superiores a 0.7 muy fuertes (Cohen, 1988).

$$V = \sqrt{\frac{\chi^2}{n \cdot \min(r-1, c-1)}}$$

El cálculo se basa en el estadístico chi-cuadrado, que mide la diferencia entre lo observado y lo esperado si las variables fueran independientes. La n representa el número total de observaciones. La r corresponde al número de filas de la tabla de contingencia y la c al número de columnas de la misma tabla.

Para que la V de Cramér sea válida se requiere que ambas variables sean categóricas, que las observaciones sean independientes y que las frecuencias esperadas de la tabla de contingencia sean adecuadas (al menos el 80 % de las celdas con frecuencia esperada ≥ 5). Cuando alguna celda queda por debajo de ese umbral, el p-valor del chi-cuadrado clásico pierde validez. En esos casos, si se requiere un contraste de hipótesis, se recomienda usar chi-cuadrado con simulación de Monte Carlo o la prueba exacta de Fisher para tablas 2×2 (Agresti, 2002). Sin embargo, la V de Cramér como medida descriptiva se calcula directamente a partir del estadístico chi-cuadrado observado, sin requerir corrección.

Una ventaja importante de la V de Cramér es que permite comparar la fuerza de asociación entre pares de variables incluso cuando las tablas de contingencia tienen dimensiones distintas, algo que no ocurre con otras medidas como el coeficiente phi, que solo es aplicable a tablas de 2x2 (Agresti, 2002). Esto resulta particularmente útil en el dataset KRKPA7, donde algunas variables son binarias y otras ternarias.
En este proyecto, la V de Cramér se utiliza con dos propósitos principales. En primer lugar, para cuantificar la relación entre cada variable predictora y la variable objetivo class, lo cual permite identificar qué atributos del tablero están más asociados con el resultado final de la partida. En segundo lugar, para construir una matriz de asociación entre todas las variables predictoras, con el fin de detectar redundancias, grupos de variables que miden conceptos similares y posibles problemas de multicolinealidad categórica que podrían afectar el desempeño de los modelos predictivos.

Residuos estandarizados: 

Una vez detectada una asociación significativa con la V de Cramér, es útil identificar qué celdas específicas contribuyen a esa asociación. Para esto se emplean los residuos estandarizados de Pearson, que comparan la frecuencia observada con la esperada bajo independencia: 
$$
r_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}
$$
Un residuo mayor a 2 en valor absoluto indica que esa celda aporta significativamente a la asociación. Un residuo mayor a 3 señala una desviación fuerte. Los residuos positivos indican que la combinación observada ocurre más de lo esperado, y los negativos que ocurre menos (Agresti, 2002). Existe también la versión ajustada de Haberman, que corrige por el tamaño de la tabla y es preferible cuando hay más de 2 filas o columnas.

## Metodología

Análisis exploratorio de datos: En el análisis univariado se examina la distribución de niveles y comportamiento individual de cada una de las 36 variables predictoras y de la variable objetivo class mediante tablas de frecuencia, gráficos de barras, y gráficos de pastel. 
En el análisis bivariado se evalúa la relación entre cada variable predictora y class mediante tablas de contingencia, matriz V de Cramér y gráficos de barras agrupados por nivel y clase. 
En el análisis multivariado se construye una matriz V de Cramér entre todas las predictoras para detectar posibles variables redundantes, grupos de variables que miden conceptos similares y posibles problemas de multicolinealidad categórica.
Las herramientas utilizadas en esta fase son R y Python, y el reporte se elabora en Bookdown y Jupyter Book respectivamente.

Visualización interactiva: Con base en los patrones identificados en el EDA, se construye un dashboard interactivo que permite filtrar por clase, es decir, por won o nowin, explorar cada variable de forma individual y agrupada, y visualizar las matrices con tooltip. Se implementa en Shiny y en Dash.

Modelado predictivo: Se procederá a construir un modelo de predicción, que, a partir de las variables del tablero, clasifique correctamente si una posición termina en victoria de las blancas (won) o no (nowin), evaluando su desempeño mediante métricas como exactitud, matriz de confusión, F1-score, AUC-ROC, y validación cruzada. Además, se implementará un chatbot en ambos dashboards.

## El dataset

El dataset KRKPA7 contiene las posiciones del final de ajedrez de 3.196 vectores de atributos que, según Shapiro (1983), clasifican las posiciones legales de este final Rey + Torre (blancas) vs Rey + Peón (negras) en la posición a7, es decir, con el peón negro a punto de coronar, y el tablero se mira desde la perspectiva del negro (Shapiro, 1983). Fueron generadas y descritas originalmente por Alen D. Shapiro en su tesis doctoral de 1983 "Structured Induction in Expert Systems". Cada posición está representada por 36 variables categóricas que describen características específicas del tablero. La variable objetivo (class) indica si el bando del peón (negras) logra ganar (won) o si las blancas lo detienen (nowin), ya sea con tablas o con victoria blanca.

Nota sobre class: UCI rotula las clases como “White-can-win” y “White-cannot-win”, pero el comportamiento de los datos indica que won corresponde al bando del peón: todas las posiciones nowin tienen rimmx = f (la torre no puede ser capturada de forma segura), y variables como el control blanco de a8 (wxqsq) o el ataque del rey blanco al peón (wkxbp) son mucho más frecuentes en nowin. Por eso, en este trabajo won se interpreta como victoria del bando del peón (negras) y nowin como que las blancas lo detienen.

Tabla de variables del dataset:


In [1]:
import pandas as pd

# Diccionario con la información de las variables (Definiciones UCI)
variables_data = {
    'Variable': [
        "wkblk", "wknwy", "wkna8", "wkona", "wkspr", "wkxbq", "wkxcr", "wkxbp",
        "whxbp", "wxqsq", "cntxt", "dsopp", "dwipd", "hdchk", "katri", "mulch",
        "qxmsq", "r2ar8", "reskd", "reskr", "rimmx", "rkxwp", "rxmsq", "simpl",
        "skach", "skewr", "skrxp", "spcop", "stlmt", "thrsk", "bkcti", "bkna8",
        "bknck", "bkovl", "bkpos", "btoeg"
    ],
    'Nombre_Real': [
        "White King in the way", "White King No Way", "White King on a8", "White King on a",
        "White King Support Rook", "White King attacked by promoted Pawn", "White King attack Critical square",
        "White King captures Black Pawn", "White attack Black Pawn", "White control Queen Square",
        "Context (edge)", "Diagonal Opposite", "Distance to intersect point", "Hidden Check",
        "King Attack Three", "Multiple Checks", "Queen Square", "Rook to A or Rank 8",
        "Reskewer Delayed", "Reskewer", "Rook in Middle", "Rook captures Pawn",
        "Rook attacks mating square", "Simple pattern", "Skewer after Checks", "Skewer",
        "Skewer or attack Pawn", "Special Opposition", "Stalemate", "Threat Skewer",
        "Black King Control The Intersect", "Black King on a8", "Black King Check",
        "Black King Overloaded", "Black King Position (potential skewer)", "Black to Egress"
    ],
    'Significado_UCI': [
        "El rey blanco obstruye el camino de la torre blanca (bloquea su movimiento).",
        "El rey blanco está en la columna 'a' o en el borde, limitando el movimiento de la torre.",
        "El rey blanco está en la casilla a8.",
        "El rey blanco está en la columna 'a'.",
        "El rey blanco puede apoyar a la torre blanca en una horquilla (fork).",
        "El rey blanco está siendo atacado por el peón promocionado.",
        "El rey blanco ataca la casilla crítica (normalmente b7).",
        "El rey blanco captura el peón negro.",
        "La torre blanca ataca al peón negro.",
        "Las blancas controlan la casilla de la dama (a8, casilla de coronación).",
        "El rey negro está en un borde del tablero.",
        "Los reyes están en oposición diagonal.",
        "La distancia del rey negro al punto de intersección es mayor que dos (g=cerca, l=lejos).",
        "Hay un jaque oculto (descubierto) contra el rey negro.",
        "Algún rey controla el punto de intersección (n=ninguno, b=negro, w=blanco).",
        "Hay múltiples jaques disponibles.",
        "La casilla de la dama (a8) está siendo atacada.",
        "La torre blanca tiene acceso a la columna 'a' o a la fila 8.",
        "Re-clavada diferida.",
        "La torre blanca puede renovar la clavada (skewer).",
        "Se puede capturar la torre blanca de forma segura.", 
        "La torre blanca captura el peón negro.",
        "La torre blanca ataca la casilla de mate.",
        "Se aplica un patrón simple (ahogado, etc).",
        "Clavada (skewer) tras jaques.",
        "Posibilidad de clavada (skewer) por parte de la torre.",
        "La torre blanca hace una clavada o ataca el peón.",
        "Oposición especial.",
        "El rey blanco está ahogado o en riesgo de estarlo.",
        "Amenaza de clavada (skewer).",
        "El rey negro controla el punto de intersección.",
        "El rey negro está en a8.",
        "El rey negro está en jaque.",
        "El rey negro está sobrecargado.",
        "El rey negro está en posición vulnerable a una clavada.",
        "El rey negro tiene vía de escape (egress)."
    ],
    'Niveles': [
        "t, f", "t, f", "t, f", "t, f", "t, f",
        "t, f", "t, f", "t, f", "t, f", "t, f",
        "t, f", "t, f", "g, l", "t, f",
        "n, w, b", "t, f", "t, f", "t, f",
        "t, f", "t, f", "t, f", "t, f",
        "t, f", "t, f", "t, f", "t, f",
        "t, f", "t, f", "t, f", "t, f",
        "t, f", "t, f", "t, f", "t, f",
        "t, f", "t, n"
    ],
    'Significado_Niveles': [
        "t=si, f=no", "t=si, f=no", "t=si, f=no", "t=si, f=no", "t=si, f=no",
        "t=si, f=no", "t=si, f=no", "t=si, f=no", "t=si, f=no", "t=si, f=no",
        "t=si (está en borde), f=no",
        "t=si, f=no",
        "g=distancia ≤ 2 (bueno para blancas), l=distancia > 2 (malo para blancas)",
        "t=si, f=no",
        "w=rey del bando de la torre (blanco), b=rey del bando del peón (negro)",
        "t=si, f=no", "t=si, f=no", "t=si, f=no",
        "t=si, f=no", "t=si, f=no", 
        "t=segura en el medio, f=expuesta/puede ser capturada", 
        "t=si, f=no",
        "t=si, f=no",
        "t=se aplica, f=no se aplica",
        "t=puede ser clavado, f=no puede",
        "t=clavada, f=no",
        "t=si, f=no",
        "t=hay, f=no hay",
        "t=hay riesgo, f=no hay riesgo",
        "t=hay amenaza, f=no hay",
        "t=si, f=no",
        "t=en a8, f=no",
        "t=jaque, f=no",
        "t=sobrecargado, f=no",
        "t=posición de skewer, f=no",
        "t=si, n=no"
    ]
}

tabla_variables = pd.DataFrame(variables_data)
tabla_variables.index = tabla_variables.index + 1

estilos = [
    {'selector': 'table', 
     'props': [('width', '100%'), ('border-collapse', 'collapse'), ('font-size', '12px')]},
    {'selector': 'th, td', 
     'props': [('padding', '6px 8px'), ('text-align', 'left'), ('white-space', 'normal'), 
               ('word-wrap', 'break-word'), ('border', '1px solid #ddd')]},
    {'selector': 'th', 
     'props': [('background-color', '#f2f2f2'), ('color', 'black'), ('font-weight', 'bold')]}
]

styled_tabla = (
    tabla_variables.style
    .set_table_styles(estilos)
    .set_properties(**{'text-align': 'left'})
    .set_properties(subset=['Variable'], **{'width': '80px', 'font-weight': 'bold'})
    .set_properties(subset=['Niveles'], **{'width': '80px', 'text-align': 'center'})
    .set_properties(subset=['Significado_Niveles'], **{'width': '150px'})
)

styled_tabla

,Variable,Nombre_Real,Significado_UCI,Niveles,Significado_Niveles
1,wkblk,White King in the way,El rey blanco obstruye el camino de la torre blanca (bloquea su movimiento).,"t, f","t=si, f=no"
2,wknwy,White King No Way,"El rey blanco está en la columna 'a' o en el borde, limitando el movimiento de la torre.","t, f","t=si, f=no"
3,wkna8,White King on a8,El rey blanco está en la casilla a8.,"t, f","t=si, f=no"
4,wkona,White King on a,El rey blanco está en la columna 'a'.,"t, f","t=si, f=no"
5,wkspr,White King Support Rook,El rey blanco puede apoyar a la torre blanca en una horquilla (fork).,"t, f","t=si, f=no"
6,wkxbq,White King attacked by promoted Pawn,El rey blanco está siendo atacado por el peón promocionado.,"t, f","t=si, f=no"
7,wkxcr,White King attack Critical square,El rey blanco ataca la casilla crítica (normalmente b7).,"t, f","t=si, f=no"
8,wkxbp,White King captures Black Pawn,El rey blanco captura el peón negro.,"t, f","t=si, f=no"
9,whxbp,White attack Black Pawn,La torre blanca ataca al peón negro.,"t, f","t=si, f=no"
10,wxqsq,White control Queen Square,"Las blancas controlan la casilla de la dama (a8, casilla de coronación).","t, f","t=si, f=no"


Observación sobre los niveles: `g` y `w` representan el mismo concepto general (acción o característica de las blancas), pero su uso depende de la variable específica:
- `w` (White): Se usa cuando la variable indica directamente una acción o posición de las blancas.
- `g` (Greater): Se usa en la variable `dwipd` para indicar que la distancia del rey negro al punto de intersección es mayor que dos.
